# 新竹交通事故風險預測（整理版）

這份 notebook 是把先前協作後的程式重新整理成一條比較一致的 pipeline，並保留必要註解，方便你之後直接在 Colab / Jupyter 上修改。

這份版本包含：

1. 參數設定  
2. 小工具函式  
3. 事故資料讀取與清理  
4. 建立 `district × date × slot_id` 面板資料  
5. 歷史事故特徵  
6. calendar 特徵  
7. weather 特徵  
8. 合併成最終建模資料  
9. train / test split  
10. 模型訓練與評估  
11. ablation 與 summary 輸出

這次特別統一了幾個先前容易打架的地方：

- 天氣特徵統一用 `weather_feat` / `weather_feat_use`
- 最終 merge 不再使用舊版的 `weather_4h`
- 模型欄位名稱統一成目前資料中真的存在的欄位
- 只保留本次分析的四個行政區：
  - 東區
  - 北區
  - 竹北市
  - 竹東鎮


## 1. 安裝與匯入套件

In [2]:
# 如果在 Colab 執行，且還沒安裝 holidays，可以取消註解這行
# !pip install holidays

import os
import re
import io
import json
import math
import glob
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import holidays

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier


## 2. 參數設定

這裡集中放所有會改的參數：

- 專案路徑
- 日期範圍
- 訓練 / 測試切分
- 分析行政區
- 時段切分
- 事故資料下載網址（若你想自動抓）

In [3]:
# =========================================================
# 參數區：請先檢查這裡
# =========================================================
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR / "hsinchu_accident_risk"

RAW_DIR = PROJECT_DIR / "data" / "raw"
OUT_DIR = PROJECT_DIR / "outputs"

ACCIDENT_DIR = RAW_DIR / "accidents"
WEATHER_DIR = RAW_DIR / "weather"
PROC_DIR = RAW_DIR / "processed"

for d in [RAW_DIR, PROC_DIR, OUT_DIR, ACCIDENT_DIR, WEATHER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 本次分析時間範圍
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

# 訓練 / 測試切分
TRAIN_END = "2024-12-31"
TEST_START = "2025-01-01"

# 僅分析四個行政區
HSINCHU_CITY_DISTRICTS = ["東區", "北區"]
HSINCHU_COUNTY_DISTRICTS = ["竹北市", "竹東鎮"]
ANALYSIS_DISTRICTS = HSINCHU_CITY_DISTRICTS + HSINCHU_COUNTY_DISTRICTS
ALL_DISTRICTS = ANALYSIS_DISTRICTS

# 每 4 小時一個時段
SLOT_LABELS = {
    0: "00:00-03:59",
    1: "04:00-07:59",
    2: "08:00-11:59",
    3: "12:00-15:59",
    4: "16:00-19:59",
    5: "20:00-23:59",
}

# 若要自動下載事故 zip，可以把網址填在這裡
ACCIDENT_URLS = []

print("PROJECT_DIR =", PROJECT_DIR)
print("RAW_DIR     =", RAW_DIR)
print("WEATHER_DIR =", WEATHER_DIR)


PROJECT_DIR = /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk
RAW_DIR     = /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/data/raw
WEATHER_DIR = /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/data/raw/weather


## 3. 小工具函式

這一段主要放一些重複會用到的小函式，例如：

- 檔案下載
- 安全讀 csv
- 行政區對應縣市
- 時段轉換
- 季節判斷
- 欄位名稱自動挑選

In [4]:
def download_file(url: str, save_path: Path, chunk_size: int = 1024 * 1024):
    """下載檔案到指定路徑。"""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)


def safe_read_csv(path, **kwargs):
    """依序嘗試常見中文編碼讀取 csv。"""
    last_err = None
    for enc in ["utf-8-sig", "cp950", "big5", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err


def infer_county_from_district(district):
    """由行政區推回新竹市 / 新竹縣。"""
    if pd.isna(district):
        return np.nan
    if district in HSINCHU_CITY_DISTRICTS:
        return "新竹市"
    if district in HSINCHU_COUNTY_DISTRICTS:
        return "新竹縣"
    return np.nan


def hour_to_slot(hour: int) -> int:
    """把小時轉成 4 小時一格的 slot_id。"""
    return int(hour) // 4


def slot_to_label(slot_id: int) -> str:
    """把 slot_id 轉成可讀標籤。"""
    return SLOT_LABELS.get(int(slot_id), "UNKNOWN")


def season_from_month(m):
    if m in [12, 1, 2]:
        return "winter"
    elif m in [3, 4, 5]:
        return "spring"
    elif m in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"


def is_peak_slot(slot_id):
    """簡單定義尖峰時段：08-12 與 16-20。"""
    return 1 if slot_id in [2, 4] else 0


def pick_first_existing(df, candidates, required=True):
    """從多個候選欄位中，選第一個存在的欄位。"""
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"找不到候選欄位：{candidates}")
    return None


## 4. 事故資料下載

如果已經手動把 ZIP 放進 `raw/accidents/`，
如果有填 `ACCIDENT_URLS`，會自動下載。

In [5]:
for i, url in enumerate(ACCIDENT_URLS, start=1):
    save_path = ACCIDENT_DIR / f"accident_source_{i}.zip"
    if save_path.exists():
        print(f"已存在，略過：{save_path.name}")
        continue
    try:
        print(f"下載中：{url[:90]}...")
        download_file(url, save_path)
    except Exception as e:
        print(f"下載失敗：{url}")
        print("原因：", e)


## 5. 讀取事故資料並清理成逐案表

這一段會：

1. 從 ZIP 中抓出 A1 / A2 事故資料  
2. 合併成一個大表  
3. 擷取日期、時間、行政區  
4. 建立 `datetime`、`slot_id`、`slot_4h`  
5. 只保留本次分析四區

In [6]:
def read_accident_zip_only_data(zip_path):
    """從單一 zip 中讀取 A1 / A2 事故資料。"""
    dfs = []

    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            base = Path(name).name

            # 只抓 A1 / A2 檔
            if not (("A1" in base) or ("A2" in base)):
                continue

            with zf.open(name) as f:
                data = f.read()

            df = None
            for enc in ["utf-8-sig", "cp950", "big5"]:
                try:
                    df = pd.read_csv(io.BytesIO(data), encoding=enc)
                    break
                except Exception:
                    continue

            if df is not None:
                df["source_zip"] = Path(zip_path).name

                # 有些原始資料最後兩列是說明文字，先去掉
                if len(df) >= 2:
                    df = df.iloc[:-2].copy()

                dfs.append(df)

    if not dfs:
        raise ValueError(f"在 {zip_path} 沒有讀到任何 A1/A2 事故資料")

    return pd.concat(dfs, ignore_index=True)


def clean_date_value(x):
    """清理日期欄，處理 20240101 或一般日期字串。"""
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    s = re.sub(r"\.0+$", "", s)

    if re.fullmatch(r"\d{8}", s):
        return pd.to_datetime(s, format="%Y%m%d", errors="coerce")

    return pd.to_datetime(s, errors="coerce")


def clean_time_value(x):
    """清理時間欄，統一轉成 HH:MM:SS。"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0+$", "", s)
    s = re.sub(r"[^\d]", "", s)

    if re.fullmatch(r"\d{6}", s):
        hh, mm, ss = int(s[:2]), int(s[2:4]), int(s[4:6])
        if 0 <= hh <= 23 and 0 <= mm <= 59 and 0 <= ss <= 59:
            return f"{hh:02d}:{mm:02d}:{ss:02d}"

    if re.fullmatch(r"\d{4}", s):
        hh, mm = int(s[:2]), int(s[2:4])
        if 0 <= hh <= 23 and 0 <= mm <= 59:
            return f"{hh:02d}:{mm:02d}:00"

    return np.nan


district_pattern = "|".join(sorted(ALL_DISTRICTS, key=len, reverse=True))


def extract_district_from_location(text):
    """從事故地點文字中抓出行政區。"""
    if pd.isna(text):
        return np.nan
    s = str(text).strip()
    m = re.search(district_pattern, s)
    return m.group(0) if m else np.nan


In [7]:
zip_files = sorted(ACCIDENT_DIR.glob("*.zip"))
print("找到的 zip：", [p.name for p in zip_files])

all_dfs = []
skipped = []

for zp in zip_files:
    try:
        df = read_accident_zip_only_data(zp)
        print(f"成功讀取: {zp.name} | shape={df.shape}")
        all_dfs.append(df)
    except Exception as e:
        print(f"跳過: {zp.name} | 原因: {e}")
        skipped.append((zp.name, str(e)))

if not all_dfs:
    raise ValueError("所有 zip 都沒有讀到 A1/A2 事故資料")

acc_raw = pd.concat(all_dfs, ignore_index=True)

# 先只保留與新竹相關的地點
acc_raw["發生地點"] = acc_raw["發生地點"].astype(str).str.strip()
acc_raw = acc_raw[acc_raw["發生地點"].str.contains("新竹", na=False)].copy()

print("acc_raw shape =", acc_raw.shape)
print("欄位數 =", len(acc_raw.columns))



找到的 zip： ['accident_source_1.zip', 'accident_source_2.zip']


/tmp/ipykernel_1912504/2163143247.py:19: DtypeWarning: Columns (0: 發生年度) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(data), encoding=enc)


成功讀取: accident_source_1.zip | shape=(888525, 52)
成功讀取: accident_source_2.zip | shape=(910164, 53)
acc_raw shape = (96373, 53)
欄位數 = 53


In [8]:
# 自動找常見欄位名稱，避免不同年份欄位名不一致
COL_DATE = pick_first_existing(acc_raw, ["發生日期", "發生年月日", "事故發生日期"])
COL_TIME = pick_first_existing(acc_raw, ["發生時間", "事故發生時間"])
COL_LOCATION = pick_first_existing(acc_raw, ["發生地點", "肇事地點", "事故地點"], required=False)
COL_LONG = pick_first_existing(acc_raw, ["經度", "經度度數", "longitude"], required=False)
COL_LAT = pick_first_existing(acc_raw, ["緯度", "緯度度數", "latitude"], required=False)
COL_WEATHER = pick_first_existing(acc_raw, ["天候名稱", "天候"], required=False)
COL_LIGHT = pick_first_existing(acc_raw, ["光線名稱", "光線"], required=False)
COL_ROADTYPE = pick_first_existing(acc_raw, ["道路型態大類別名稱", "道路型態名稱"], required=False)
COL_SPEED = pick_first_existing(acc_raw, ["速限-第1當事者", "速限"], required=False)
COL_CASUALTIES = pick_first_existing(acc_raw, ["死亡受傷人數", "死傷人數"], required=False)

print("COL_DATE     =", COL_DATE)
print("COL_TIME     =", COL_TIME)
print("COL_LOCATION =", COL_LOCATION)


COL_DATE     = 發生日期
COL_TIME     = 發生時間
COL_LOCATION = 發生地點


In [9]:
acc = acc_raw.copy()

# 建立標準化日期時間欄位
acc["date"] = acc[COL_DATE].apply(clean_date_value)
acc["time_str"] = acc[COL_TIME].apply(clean_time_value)
acc["datetime"] = pd.to_datetime(
    acc["date"].dt.strftime("%Y-%m-%d") + " " + acc["time_str"].astype(str),
    errors="coerce"
)

# 擷取行政區與衍生欄位
acc["district"] = acc[COL_LOCATION].apply(extract_district_from_location)
acc["county_city"] = acc["district"].apply(infer_county_from_district)
acc["hour"] = acc["datetime"].dt.hour
acc["slot_id"] = acc["hour"].apply(lambda x: hour_to_slot(x) if pd.notna(x) else np.nan)
acc["slot_4h"] = acc["slot_id"].apply(lambda x: slot_to_label(x) if pd.notna(x) else np.nan)

# 座標欄若存在就保留
acc["longitude"] = pd.to_numeric(acc[COL_LONG], errors="coerce") if COL_LONG else np.nan
acc["latitude"] = pd.to_numeric(acc[COL_LAT], errors="coerce") if COL_LAT else np.nan

# 保留之後可能會用到的欄位
keep_cols = [
    "datetime", "date", "time_str", "county_city", "district",
    "hour", "slot_id", "slot_4h", "longitude", "latitude"
]

for c_src, c_new in [
    (COL_LOCATION, "location_text"),
    (COL_WEATHER, "accident_weather"),
    (COL_LIGHT, "light"),
    (COL_ROADTYPE, "road_type"),
    (COL_SPEED, "speed_limit"),
    (COL_CASUALTIES, "casualties"),
]:
    if c_src is not None and c_src in acc.columns:
        acc[c_new] = acc[c_src]
        keep_cols.append(c_new)

acc = acc[keep_cols].copy()

# 依分析期間與行政區篩選
acc = acc[(acc["date"] >= pd.to_datetime(START_DATE)) & (acc["date"] <= pd.to_datetime(END_DATE))]
acc = acc.dropna(subset=["datetime", "district", "slot_id"])
acc = acc[acc["district"].isin(ANALYSIS_DISTRICTS)].copy()

print("事故清理後 shape =", acc.shape)
display(acc.head())
print(acc["district"].value_counts())


事故清理後 shape = (44927, 16)


,datetime,date,time_str,county_city,district,hour,slot_id,slot_4h,longitude,latitude,location_text,accident_weather,light,road_type,speed_limit,casualties
54,2024-01-07 11:19:00,2024-01-07,11:19:00,新竹市,東區,11.0,2.0,08:00-11:59,120.979653,24.789102,新竹市東區寶山路247號前0.0公尺附近,晴,有照明未開啟或故障,單路部分,50.0,死亡1;受傷0
55,2024-01-07 11:19:00,2024-01-07,11:19:00,新竹市,東區,11.0,2.0,08:00-11:59,120.979653,24.789102,新竹市東區寶山路247號前0.0公尺附近,晴,有照明未開啟或故障,單路部分,50.0,死亡1;受傷0
484,2024-02-06 18:17:28,2024-02-06,18:17:28,新竹縣,竹北市,18.0,4.0,16:00-19:59,120.940824,24.849738,新竹縣竹北市西濱路一段台15線73K附近,晴,無照明,單路部分,60.0,死亡1;受傷1
485,2024-02-06 18:17:28,2024-02-06,18:17:28,新竹縣,竹北市,18.0,4.0,16:00-19:59,120.940824,24.849738,新竹縣竹北市西濱路一段台15線73K附近,晴,無照明,單路部分,60.0,死亡1;受傷1
543,2024-02-11 13:40:00,2024-02-11,13:40:00,新竹市,東區,13.0,3.0,12:00-15:59,121.010213,24.787880,新竹市東區慈雲路(533064燈桿),晴,有照明且開啟,單路部分,40.0,死亡1;受傷1


district
東區     18298
竹北市    14143
北區      7986
竹東鎮     4500
Name: count, dtype: int64


In [10]:
acc.to_csv(PROC_DIR / "accidents_clean.csv", index=False, encoding="utf-8-sig")
print("已輸出：", PROC_DIR / "accidents_clean.csv")


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/data/raw/processed/accidents_clean.csv


## 6. 建立 `district × date × 4h slot` 完整 panel

建立完整樣本空間：
- 每一天
- 每個行政區
- 每個 4 小時時段

沒有事故的格子也會保留下來，才能做為分類模型label=0的資料。

In [11]:
date_range = pd.date_range(START_DATE, END_DATE, freq="D")

base = (
    pd.MultiIndex.from_product(
        [date_range, ANALYSIS_DISTRICTS, list(SLOT_LABELS.keys())],
        names=["date", "district", "slot_id"]
    )
    .to_frame(index=False)
)

base["county_city"] = base["district"].apply(infer_county_from_district)
base["slot_4h"] = base["slot_id"].map(SLOT_LABELS)
base["date"] = pd.to_datetime(base["date"])

# 把事故逐案資料聚合成每格的事故數
agg_acc = (
    acc.groupby(["date", "district", "slot_id"], as_index=False)
       .size()
       .rename(columns={"size": "accident_count"})
)

panel = base.merge(agg_acc, on=["date", "district", "slot_id"], how="left")
panel["accident_count"] = panel["accident_count"].fillna(0).astype(int)

# label = 當格是否有事故
panel["label"] = (panel["accident_count"] > 0).astype(int)

print("panel shape =", panel.shape)
display(panel.head())


panel shape = (17544, 7)


,date,district,slot_id,county_city,slot_4h,accident_count,label
0,2024-01-01,東區,0,新竹市,00:00-03:59,0,0
1,2024-01-01,東區,1,新竹市,04:00-07:59,0,0
2,2024-01-01,東區,2,新竹市,08:00-11:59,7,1
3,2024-01-01,東區,3,新竹市,12:00-15:59,9,1
4,2024-01-01,東區,4,新竹市,16:00-19:59,10,1


## 7. 歷史事故特徵

這一段建立兩種常見歷史特徵：

1. **過去幾天的總事故數**  
2. **同一行政區、同一時段，過去幾天的事故數**

In [12]:
history_windows = [1, 3, 5, 7]

panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["district", "date", "slot_id"]).reset_index(drop=True)

# 先做「每日總事故數」
daily_acc = (
    panel.groupby(["district", "date"], as_index=False)["accident_count"]
         .sum()
         .rename(columns={"accident_count": "daily_accidents"})
         .sort_values(["district", "date"])
         .reset_index(drop=True)
)

for w in history_windows:
    col = f"past_{w}day_accidents"
    daily_acc[col] = (
        daily_acc.groupby("district")["daily_accidents"]
                 .transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
    )

merge_cols = ["district", "date"] + [f"past_{w}day_accidents" for w in history_windows]
panel = panel.merge(daily_acc[merge_cols], on=["district", "date"], how="left")

# 再做「同一時段」的歷史事故數
for w in history_windows:
    col = f"past_same_slot_{w}day_accidents"
    panel = panel.sort_values(["district", "slot_id", "date"]).reset_index(drop=True)
    panel[col] = (
        panel.groupby(["district", "slot_id"])["accident_count"]
             .transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
    )

history_cols = [f"past_{w}day_accidents" for w in history_windows] +                [f"past_same_slot_{w}day_accidents" for w in history_windows]
panel[history_cols] = panel[history_cols].fillna(0)

display(panel.head())


,date,district,slot_id,county_city,slot_4h,accident_count,label,past_1day_accidents,past_3day_accidents,past_5day_accidents,past_7day_accidents,past_same_slot_1day_accidents,past_same_slot_3day_accidents,past_same_slot_5day_accidents,past_same_slot_7day_accidents
0,2024-01-01,北區,0,新竹市,00:00-03:59,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-01-02,北區,0,新竹市,00:00-03:59,0,0,15.0,15.0,15.0,15.0,0.0,0.0,0.0,0.0
2,2024-01-03,北區,0,新竹市,00:00-03:59,0,0,12.0,27.0,27.0,27.0,0.0,0.0,0.0,0.0
3,2024-01-04,北區,0,新竹市,00:00-03:59,0,0,16.0,43.0,43.0,43.0,0.0,0.0,0.0,0.0
4,2024-01-05,北區,0,新竹市,00:00-03:59,0,0,4.0,32.0,47.0,47.0,0.0,0.0,0.0,0.0


## 8. 建立 calendar 特徵

calendar 特徵是和行政區無關、只跟日期有關的特徵，例如：

- 星期幾
- 月份
- 季節
- 是否週末
- 是否國定假日
- 是否一般上班日

In [13]:
def build_calendar_features(start_date, end_date):
    dates = pd.DataFrame({"date": pd.date_range(start_date, end_date, freq="D")})
    dates["weekday"] = dates["date"].dt.weekday
    dates["month"] = dates["date"].dt.month
    dates["day"] = dates["date"].dt.day
    dates["quarter"] = dates["date"].dt.quarter
    dates["season"] = dates["month"].apply(season_from_month)
    dates["is_weekend"] = (dates["weekday"] >= 5).astype(int)

    years = sorted(dates["date"].dt.year.unique().tolist())
    tw_holidays = holidays.TW(years=years)

    dates["holiday_name"] = dates["date"].dt.date.map(lambda d: tw_holidays.get(d, ""))
    dates["is_holiday"] = dates["date"].dt.date.map(lambda d: int(d in tw_holidays))
    dates["is_workday"] = ((1 - dates["is_weekend"]) & (1 - dates["is_holiday"])).astype(int)
    return dates

calendar_df = build_calendar_features(START_DATE, END_DATE)

print("calendar_df shape =", calendar_df.shape)
display(calendar_df.head())


calendar_df shape = (731, 10)


,date,weekday,month,day,quarter,season,is_weekend,holiday_name,is_holiday,is_workday
0,2024-01-01,0,1,1,1,winter,0,Founding Day of the Republic of China,1,0
1,2024-01-02,1,1,2,1,winter,0,,0,1
2,2024-01-03,2,1,3,1,winter,0,,0,1
3,2024-01-04,3,1,4,1,winter,0,,0,1
4,2024-01-05,4,1,5,1,winter,0,,0,1


## 9. 讀取 weather 資料

天氣資料放在：

- `raw/weather/新竹市東區/`
- `raw/weather/新竹市北區/`
- `raw/weather/新竹縣竹東鎮/`
- `raw/weather/新竹縣竹北市/` 或 `raw/weather/新竹市竹北市/`

每個月一個 csv，檔名像：
- `COD560-2025-02.csv`

可以直接從檔名抓年 / 月。

In [14]:
district_map = {
    "新竹市東區": "東區",
    "新竹市北區": "北區",
    "新竹縣竹東鎮": "竹東鎮",
    "新竹縣竹北市": "竹北市",
    "新竹市竹北市": "竹北市",  # 若你實際資料夾名稱是這個，也能兼容
}

rename_map = {
    "ObsTime": "day",
    "StnPres": "stn_pres",
    "StnPresMax": "stn_pres_max",
    "StnPresMaxTime": "stn_pres_max_time",
    "StnPresMin": "stn_pres_min",
    "StnPresMinTime": "stn_pres_min_time",
    "Temperature": "temp_avg",
    "T Max": "temp_max",
    "T Max Time": "temp_max_time",
    "T Min": "temp_min",
    "T Min Time": "temp_min_time",
    "RH": "rh_avg",
    "RHMin": "rh_min",
    "RHMinTime": "rh_min_time",
    "WS": "wind_speed",
    "WD": "wind_dir",
    "WSGust": "wind_gust",
    "WDGust": "wind_gust_dir",
    "WGustTime": "wind_gust_time",
    "Precp": "precip",
}

numeric_cols = [
    "stn_pres", "stn_pres_max", "stn_pres_min",
    "temp_avg", "temp_max", "temp_min",
    "rh_avg", "rh_min",
    "wind_speed", "wind_dir",
    "wind_gust", "wind_gust_dir",
    "precip",
]


def read_weather_file(file_path: Path, district: str) -> pd.DataFrame:
    """讀取單一月天氣檔，並由檔名解析 year / month。"""
    try:
        raw = pd.read_csv(file_path, header=None, encoding="utf-8-sig")
    except UnicodeDecodeError:
        raw = pd.read_csv(file_path, header=None, encoding="big5")
    except Exception:
        raw = pd.read_csv(file_path, header=None)

    # 第 1 列：中文欄名；第 2 列：英文欄名；第 3 列開始：資料
    raw.columns = raw.iloc[1]
    df = raw.iloc[2:].copy()

    keep_cols = [c for c in rename_map.keys() if c in df.columns]
    df = df[keep_cols].rename(columns=rename_map)

    # 例如：COD560-2025-02.csv -> year=2025, month=2
    m = re.search(r"(\d{4})-(\d{2})", file_path.name)
    if not m:
        raise ValueError(f"檔名無法解析年月: {file_path.name}")

    year = int(m.group(1))
    month = int(m.group(2))

    df["year"] = year
    df["month"] = month
    df["district"] = district
    df["day"] = pd.to_numeric(df["day"], errors="coerce")
    df["date"] = pd.to_datetime(
        dict(year=df["year"], month=df["month"], day=df["day"]),
        errors="coerce"
    )

    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


In [15]:
all_weather = []

for folder_name, district in district_map.items():
    district_dir = WEATHER_DIR / folder_name
    if not district_dir.exists():
        continue

    csv_files = sorted(district_dir.glob("*.csv"))
    print(f"{folder_name} -> {district}: {len(csv_files)} files")

    for f in csv_files:
        try:
            all_weather.append(read_weather_file(f, district))
        except Exception as e:
            print(f"讀取失敗: {f} | 原因: {e}")

if not all_weather:
    raise ValueError("沒有成功讀到 weather 資料")

weather_df = pd.concat(all_weather, ignore_index=True)
weather_df = weather_df[weather_df["district"].isin(ANALYSIS_DISTRICTS)].copy()
weather_df = weather_df.sort_values(["district", "date"]).dropna(subset=["district", "date"]).reset_index(drop=True)

print("weather_df shape =", weather_df.shape)
display(weather_df.head())
print(weather_df["district"].value_counts())


新竹市東區 -> 東區: 27 files
新竹市北區 -> 北區: 27 files
新竹縣竹東鎮 -> 竹東鎮: 27 files
新竹縣竹北市 -> 竹北市: 27 files
weather_df shape = (3254, 24)


1,day,stn_pres,stn_pres_max,stn_pres_max_time,stn_pres_min,stn_pres_min_time,temp_avg,temp_max,temp_max_time,temp_min,...,wind_speed,wind_dir,wind_gust,wind_gust_dir,wind_gust_time,precip,year,month,district,date
0,1,1019.2,1021.3,2024/01/01 09:27:00,1017.7,2024/01/01 15:05:00,16.9,19.3,2024/01/01 11:57:00,15.8,...,5.5,40.0,14.0,52.0,2024/01/01 06:20:00,0.0,2024,1,北區,2024-01-01
1,2,1016.3,1018.6,2024/01/02 01:10:00,1013.6,2024/01/02 14:30:00,17.4,21.5,2024/01/02 10:45:00,11.8,...,3.2,23.0,12.9,24.0,2024/01/02 12:54:00,0.0,2024,1,北區,2024-01-02
2,3,1019.1,1021.4,2024/01/03 20:51:00,1015.6,2024/01/03 02:51:00,15.8,18.5,2024/01/03 03:44:00,13.1,...,5.5,49.0,15.0,59.0,2024/01/03 23:08:00,0.0,2024,1,北區,2024-01-03
3,4,1018.4,1020.6,2024/01/04 00:01:00,1016.3,2024/01/04 14:27:00,15.9,19.9,2024/01/04 14:01:00,12.8,...,4.1,53.0,14.4,42.0,2024/01/04 00:13:00,0.0,2024,1,北區,2024-01-04
4,5,1018.7,1020.6,2024/01/05 08:54:00,1016.8,2024/01/05 13:59:00,16.5,20.1,2024/01/05 10:37:00,13.8,...,2.4,352.0,7.8,346.0,2024/01/05 13:48:00,0.0,2024,1,北區,2024-01-05


district
北區     821
東區     821
竹北市    821
竹東鎮    791
Name: count, dtype: int64


## 10. 建立 weather 特徵

- 原始天氣欄位  
- 衍生欄位（溫差、是否下雨、是否高溫等）  
- lag 特徵  
- rolling 特徵  
- 前 3 天累積雨量

In [16]:
weather_feat = weather_df.copy()
weather_feat["date"] = pd.to_datetime(weather_feat["date"], errors="coerce")

# 衍生特徵
weather_feat["temp_range"] = weather_feat["temp_max"] - weather_feat["temp_min"]
weather_feat["gust_gap"] = weather_feat["wind_gust"] - weather_feat["wind_speed"]
weather_feat["stn_pres_range"] = weather_feat["stn_pres_max"] - weather_feat["stn_pres_min"]

weather_feat["is_rain"] = (weather_feat["precip"] > 0).astype(int)
weather_feat["is_heavy_rain"] = (weather_feat["precip"] >= 25).astype(int)
weather_feat["is_hot_day"] = (weather_feat["temp_max"] >= 30).astype(int)
weather_feat["is_cold_day"] = (weather_feat["temp_min"] <= 10).astype(int)
weather_feat["is_humid"] = (weather_feat["rh_avg"] >= 85).astype(int)
weather_feat["is_strong_wind"] = (weather_feat["wind_gust"] >= 10).astype(int)

# lag 特徵：昨天的天氣
lag_cols = ["temp_avg", "rh_avg", "wind_speed", "wind_gust", "precip", "stn_pres"]
for col in lag_cols:
    weather_feat[f"{col}_lag1"] = weather_feat.groupby("district")[col].shift(1)

# rolling 平均：只用昨天以前資料，避免 leakage
roll_cols = ["temp_avg", "rh_avg", "wind_speed", "wind_gust", "precip", "stn_pres"]
for col in roll_cols:
    weather_feat[f"{col}_roll3_mean"] = (
        weather_feat.groupby("district")[col]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    )
    weather_feat[f"{col}_roll7_mean"] = (
        weather_feat.groupby("district")[col]
        .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
    )

weather_feat["precip_sum_3d"] = (
    weather_feat.groupby("district")["precip"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
)

print("weather_feat shape =", weather_feat.shape)
display(weather_feat.head())


weather_feat shape = (3254, 52)


1,day,stn_pres,stn_pres_max,stn_pres_max_time,stn_pres_min,stn_pres_min_time,temp_avg,temp_max,temp_max_time,temp_min,...,rh_avg_roll7_mean,wind_speed_roll3_mean,wind_speed_roll7_mean,wind_gust_roll3_mean,wind_gust_roll7_mean,precip_roll3_mean,precip_roll7_mean,stn_pres_roll3_mean,stn_pres_roll7_mean,precip_sum_3d
0,1,1019.2,1021.3,2024/01/01 09:27:00,1017.7,2024/01/01 15:05:00,16.9,19.3,2024/01/01 11:57:00,15.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1016.3,1018.6,2024/01/02 01:10:00,1013.6,2024/01/02 14:30:00,17.4,21.5,2024/01/02 10:45:00,11.8,...,66.000000,5.500000,5.500000,14.000000,14.000000,0.0,0.0,1019.200000,1019.20,0.0
2,3,1019.1,1021.4,2024/01/03 20:51:00,1015.6,2024/01/03 02:51:00,15.8,18.5,2024/01/03 03:44:00,13.1,...,72.500000,4.350000,4.350000,13.450000,13.450000,0.0,0.0,1017.750000,1017.75,0.0
3,4,1018.4,1020.6,2024/01/04 00:01:00,1016.3,2024/01/04 14:27:00,15.9,19.9,2024/01/04 14:01:00,12.8,...,74.666667,4.733333,4.733333,13.966667,13.966667,0.0,0.0,1018.200000,1018.20,0.0
4,5,1018.7,1020.6,2024/01/05 08:54:00,1016.8,2024/01/05 13:59:00,16.5,20.1,2024/01/05 10:37:00,13.8,...,72.250000,4.266667,4.575000,14.100000,14.075000,0.0,0.0,1017.933333,1018.25,0.0


In [17]:
# 最後 merge 用的乾淨天氣表
weather_keep = [
    "district", "date",
    "temp_avg", "temp_max", "temp_min", "temp_range",
    "rh_avg", "rh_min",
    "wind_speed", "wind_gust", "gust_gap",
    "stn_pres", "stn_pres_range",
    "precip", "is_rain", "is_heavy_rain",
    "is_hot_day", "is_cold_day", "is_humid", "is_strong_wind",
    "temp_avg_lag1", "rh_avg_lag1", "wind_speed_lag1", "precip_lag1",
    "temp_avg_roll3_mean", "temp_avg_roll7_mean",
    "rh_avg_roll3_mean", "rh_avg_roll7_mean",
    "wind_speed_roll3_mean", "wind_speed_roll7_mean",
    "precip_roll3_mean", "precip_roll7_mean",
    "precip_sum_3d",
]
weather_keep = [c for c in weather_keep if c in weather_feat.columns]

weather_feat_use = weather_feat[weather_keep].drop_duplicates(["district", "date"]).copy()

print("weather_feat_use shape =", weather_feat_use.shape)
display(weather_feat_use.head())


weather_feat_use shape = (3254, 33)


1,district,date,temp_avg,temp_max,temp_min,temp_range,rh_avg,rh_min,wind_speed,wind_gust,...,precip_lag1,temp_avg_roll3_mean,temp_avg_roll7_mean,rh_avg_roll3_mean,rh_avg_roll7_mean,wind_speed_roll3_mean,wind_speed_roll7_mean,precip_roll3_mean,precip_roll7_mean,precip_sum_3d
0,北區,2024-01-01,16.9,19.3,15.8,3.5,66.0,57.0,5.5,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,北區,2024-01-02,17.4,21.5,11.8,9.7,79.0,59.0,3.2,12.9,...,0.0,16.900000,16.90,66.000000,66.000000,5.500000,5.500000,0.0,0.0,0.0
2,北區,2024-01-03,15.8,18.5,13.1,5.4,79.0,60.0,5.5,15.0,...,0.0,17.150000,17.15,72.500000,72.500000,4.350000,4.350000,0.0,0.0,0.0
3,北區,2024-01-04,15.9,19.9,12.8,7.1,65.0,53.0,4.1,14.4,...,0.0,16.700000,16.70,74.666667,74.666667,4.733333,4.733333,0.0,0.0,0.0
4,北區,2024-01-05,16.5,20.1,13.8,6.3,85.0,76.0,2.4,7.8,...,0.0,16.366667,16.50,74.333333,72.250000,4.266667,4.575000,0.0,0.0,0.0


## 11. 合併 `panel + calendar_df + weather_feat`
最終合併分析資料的大表
- `calendar_df` 用 `date` merge
- `weather_feat_use` 用 `["district", "date"]` merge



In [18]:
panel["is_peak_hour"] = panel["slot_id"].apply(is_peak_slot)

calendar_keep = [
    "date", "weekday", "month", "day", "quarter",
    "season", "is_weekend", "is_holiday", "is_workday", "holiday_name"
]
calendar_df_use = calendar_df[calendar_keep].drop_duplicates("date").copy()

final_df = (
    panel
    .merge(calendar_df_use, on="date", how="left")
    .merge(weather_feat_use, on=["district", "date"], how="left")
)

print("final_df shape =", final_df.shape)
display(final_df.head())
print(final_df.columns.tolist())


final_df shape = (17544, 56)


,date,district,slot_id,county_city,slot_4h,accident_count,label,past_1day_accidents,past_3day_accidents,past_5day_accidents,...,precip_lag1,temp_avg_roll3_mean,temp_avg_roll7_mean,rh_avg_roll3_mean,rh_avg_roll7_mean,wind_speed_roll3_mean,wind_speed_roll7_mean,precip_roll3_mean,precip_roll7_mean,precip_sum_3d
0,2024-01-01,北區,0,新竹市,00:00-03:59,0,0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-02,北區,0,新竹市,00:00-03:59,0,0,15.0,15.0,15.0,...,0.0,16.900000,16.90,66.000000,66.000000,5.500000,5.500000,0.0,0.0,0.0
2,2024-01-03,北區,0,新竹市,00:00-03:59,0,0,12.0,27.0,27.0,...,0.0,17.150000,17.15,72.500000,72.500000,4.350000,4.350000,0.0,0.0,0.0
3,2024-01-04,北區,0,新竹市,00:00-03:59,0,0,16.0,43.0,43.0,...,0.0,16.700000,16.70,74.666667,74.666667,4.733333,4.733333,0.0,0.0,0.0
4,2024-01-05,北區,0,新竹市,00:00-03:59,0,0,4.0,32.0,47.0,...,0.0,16.366667,16.50,74.333333,72.250000,4.266667,4.575000,0.0,0.0,0.0


['date', 'district', 'slot_id', 'county_city', 'slot_4h', 'accident_count', 'label', 'past_1day_accidents', 'past_3day_accidents', 'past_5day_accidents', 'past_7day_accidents', 'past_same_slot_1day_accidents', 'past_same_slot_3day_accidents', 'past_same_slot_5day_accidents', 'past_same_slot_7day_accidents', 'is_peak_hour', 'weekday', 'month', 'day', 'quarter', 'season', 'is_weekend', 'is_holiday', 'is_workday', 'holiday_name', 'temp_avg', 'temp_max', 'temp_min', 'temp_range', 'rh_avg', 'rh_min', 'wind_speed', 'wind_gust', 'gust_gap', 'stn_pres', 'stn_pres_range', 'precip', 'is_rain', 'is_heavy_rain', 'is_hot_day', 'is_cold_day', 'is_humid', 'is_strong_wind', 'temp_avg_lag1', 'rh_avg_lag1', 'wind_speed_lag1', 'precip_lag1', 'temp_avg_roll3_mean', 'temp_avg_roll7_mean', 'rh_avg_roll3_mean', 'rh_avg_roll7_mean', 'wind_speed_roll3_mean', 'wind_speed_roll7_mean', 'precip_roll3_mean', 'precip_roll7_mean', 'precip_sum_3d']


In [19]:
final_path = PROC_DIR / "final_dataset.csv"
final_df.to_csv(final_path, index=False, encoding="utf-8-sig")
print("已輸出：", final_path)


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/data/raw/processed/final_dataset.csv


## 12. 訓練 / 測試切分

目前設定：

- train：2024-01-01 ~ 2024-12-31
- test：2025-01-01 ~ 2025-12-31

後續所有實驗都以這個切分為基礎：

1. **在 train 上做時間序列 cross-validation**（避免未來資訊洩漏）
2. **在 test 上做最終 holdout evaluation**（模擬真正往未來預測）

In [20]:
train_df = final_df[final_df['date'] <= pd.to_datetime(TRAIN_END)].copy()
test_df = final_df[final_df['date'] >= pd.to_datetime(TEST_START)].copy()

print('train:', train_df.shape, train_df['date'].min(), '->', train_df['date'].max())
print('test :', test_df.shape, test_df['date'].min(), '->', test_df['date'].max())

print('train positive rate =', round(train_df['label'].mean(), 4))
print('test  positive rate =', round(test_df['label'].mean(), 4))

train: (8784, 56) 2024-01-01 00:00:00 -> 2024-12-31 00:00:00
test : (8760, 56) 2025-01-01 00:00:00 -> 2025-12-31 00:00:00
train positive rate = 0.4791
test  positive rate = 0.4748


## 13. 建模設定與共用函式

實作四組實驗：

1. **主模型比較**：Logistic Regression、XGBoost  
2. **特徵 ablation**：Time-only / Time + Weather / Time + Weather + History  
3. **類別不平衡處理**：
   - Logistic Regression without balancing
   - Logistic Regression + `class_weight='balanced'`
   - XGBoost without balancing
   - XGBoost + `scale_pos_weight`（註：XGBoost 並沒有 `class_weight='balanced_subsample'` 這個參數，因此改用功能等價、專門處理正負樣本不平衡的 `scale_pos_weight`）
4. **資料量實驗**：使用訓練集前段時間資料的 25% / 50% / 75% / 100%

評估指標統一只保留：
- **F1**
- **AUROC**

In [21]:
# ---------------------------------------------------------
# 特徵群定義
# ---------------------------------------------------------
TIME_FEATURES = [
    "county_city", "district", "slot_id",
    "is_weekend", "is_holiday", "month", "season", "is_peak_hour"
]

WEATHER_FEATURES = [
    "temp_avg", "rh_avg", "precip", "wind_speed",
    "temp_range", "gust_gap", "is_rain",
    "temp_avg_lag1", "precip_lag1", "precip_sum_3d"
]

HISTORY_FEATURES = [
    "past_1day_accidents", "past_3day_accidents",
    "past_5day_accidents", "past_7day_accidents",
    "past_same_slot_1day_accidents", "past_same_slot_3day_accidents",
    "past_same_slot_5day_accidents", "past_same_slot_7day_accidents"
]

TIME_FEATURES = [c for c in TIME_FEATURES if c in final_df.columns]
WEATHER_FEATURES = [c for c in WEATHER_FEATURES if c in final_df.columns]
HISTORY_FEATURES = [c for c in HISTORY_FEATURES if c in final_df.columns]

def split_feature_types(feature_list):
    categorical_candidates = ["county_city", "district", "season"]
    categorical = [c for c in categorical_candidates if c in feature_list]
    numeric = [c for c in feature_list if c not in categorical]
    return categorical, numeric


def build_preprocessor(feature_list, scale_numeric=True):
    categorical_features, numeric_features = split_feature_types(feature_list)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )
    return preprocessor


def compute_scale_pos_weight(y):
    y = pd.Series(y)
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    if pos == 0:
        return 1.0
    return neg / pos


def build_lr_pipeline(feature_list, class_weight=None):
    preprocessor = build_preprocessor(feature_list, scale_numeric=True)
    model = LogisticRegression(
        max_iter=2000,
        class_weight=class_weight,
        solver="lbfgs",
        random_state=42,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def build_xgb_pipeline(feature_list, use_balancing=False):
    preprocessor = build_preprocessor(feature_list, scale_numeric=False)
    scale_pos_weight = compute_scale_pos_weight(y_train_full) if use_balancing else 1.0

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def get_pred_and_score(model, X):
    pred = model.predict(X)
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        score = model.decision_function(X)
    else:
        score = pred
    return pred, score


def calc_metrics(y_true, pred, score):
    out = {
        "f1": f1_score(y_true, pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        out["auroc"] = roc_auc_score(y_true, score)
    else:
        out["auroc"] = np.nan
    return out


def time_series_cv_eval(model_builder, feature_list, train_df, target_col="label", n_splits=5):
    df = train_df.sort_values(["date", "district", "slot_id"]).reset_index(drop=True)
    X = df[feature_list].copy()
    y = df[target_col].copy()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(tscv.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = model_builder(feature_list)
        model.fit(X_tr, y_tr)
        pred_va, score_va = get_pred_and_score(model, X_va)
        m = calc_metrics(y_va, pred_va, score_va)
        m.update({
            "fold": fold_id,
            "train_start": str(df.iloc[tr_idx]["date"].min().date()),
            "train_end": str(df.iloc[tr_idx]["date"].max().date()),
            "valid_start": str(df.iloc[va_idx]["date"].min().date()),
            "valid_end": str(df.iloc[va_idx]["date"].max().date()),
            "train_rows": int(len(tr_idx)),
            "valid_rows": int(len(va_idx)),
        })
        fold_rows.append(m)

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "cv_f1_mean": fold_df["f1"].mean(),
        "cv_f1_std": fold_df["f1"].std(ddof=0),
        "cv_auroc_mean": fold_df["auroc"].mean(),
        "cv_auroc_std": fold_df["auroc"].std(ddof=0),
    }
    return fold_df, summary


def holdout_eval(model_builder, feature_list, train_df, test_df, target_col="label"):
    X_train = train_df[feature_list].copy()
    y_train = train_df[target_col].copy()
    X_test = test_df[feature_list].copy()
    y_test = test_df[target_col].copy()

    model = model_builder(feature_list)
    model.fit(X_train, y_train)
    pred_test, score_test = get_pred_and_score(model, X_test)
    metrics = calc_metrics(y_test, pred_test, score_test)
    return metrics, model


def run_experiment_suite(model_dict, feature_sets_dict, train_df, test_df, output_prefix):
    summary_rows = []
    fold_dfs = []

    for model_name, model_builder in model_dict.items():
        for feat_name, feat_cols in feature_sets_dict.items():
            cv_fold_df, cv_summary = time_series_cv_eval(
                model_builder=model_builder,
                feature_list=feat_cols,
                train_df=train_df,
                target_col="label",
                n_splits=5,
            )
            holdout_metrics, _ = holdout_eval(
                model_builder=model_builder,
                feature_list=feat_cols,
                train_df=train_df,
                test_df=test_df,
                target_col="label",
            )

            row = {
                "experiment": output_prefix,
                "model": model_name,
                "feature_set": feat_name,
                "n_features": len(feat_cols),
                **cv_summary,
                "test_f1": holdout_metrics["f1"],
                "test_auroc": holdout_metrics["auroc"],
            }
            summary_rows.append(row)

            cv_fold_df = cv_fold_df.copy()
            cv_fold_df["experiment"] = output_prefix
            cv_fold_df["model"] = model_name
            cv_fold_df["feature_set"] = feat_name
            fold_dfs.append(cv_fold_df)

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["test_f1", "test_auroc"], ascending=False
    ).reset_index(drop=True)
    fold_df_all = pd.concat(fold_dfs, ignore_index=True) if fold_dfs else pd.DataFrame()
    return summary_df, fold_df_all


y_train_full = train_df["label"].copy()

feature_sets = {
    "time_only": TIME_FEATURES,
    "time_weather": TIME_FEATURES + WEATHER_FEATURES,
    "time_weather_history": TIME_FEATURES + WEATHER_FEATURES + HISTORY_FEATURES,
}

print("TIME_FEATURES:", TIME_FEATURES)
print("WEATHER_FEATURES:", WEATHER_FEATURES)
print("HISTORY_FEATURES:", HISTORY_FEATURES)
print({k: len(v) for k, v in feature_sets.items()})

TIME_FEATURES: ['county_city', 'district', 'slot_id', 'is_weekend', 'is_holiday', 'month', 'season', 'is_peak_hour']
WEATHER_FEATURES: ['temp_avg', 'rh_avg', 'precip', 'wind_speed', 'temp_range', 'gust_gap', 'is_rain', 'temp_avg_lag1', 'precip_lag1', 'precip_sum_3d']
HISTORY_FEATURES: ['past_1day_accidents', 'past_3day_accidents', 'past_5day_accidents', 'past_7day_accidents', 'past_same_slot_1day_accidents', 'past_same_slot_3day_accidents', 'past_same_slot_5day_accidents', 'past_same_slot_7day_accidents']
{'time_only': 8, 'time_weather': 18, 'time_weather_history': 26}


## 14. 實驗 1：主模型比較

主模型為：

- **Logistic Regression**
- **XGBoost**

這裡固定使用最完整的特徵組：
- `Time + Weather + History`

並同時回報：
- train 上的 **TimeSeriesSplit cross-validation**
- 2025 test 的 **F1 / AUROC**

In [22]:
main_feature_sets = {
    "time_weather_history": feature_sets["time_weather_history"]
}

main_models = {
    "LogisticRegression": lambda feats: build_lr_pipeline(feats, class_weight=None),
    "XGBoost": lambda feats: build_xgb_pipeline(feats, use_balancing=False),
}

main_results_df, main_cv_folds_df = run_experiment_suite(
    model_dict=main_models,
    feature_sets_dict=main_feature_sets,
    train_df=train_df,
    test_df=test_df,
    output_prefix="main_model_comparison",
)

main_results_path = OUT_DIR / "exp1_main_model_comparison.csv"
main_cv_path = OUT_DIR / "exp1_main_model_comparison_cv_folds.csv"
main_results_df.to_csv(main_results_path, index=False, encoding="utf-8-sig")
main_cv_folds_df.to_csv(main_cv_path, index=False, encoding="utf-8-sig")

display(main_results_df)
print("已輸出：", main_results_path)
print("已輸出：", main_cv_path)

,experiment,model,feature_set,n_features,cv_f1_mean,cv_f1_std,cv_auroc_mean,cv_auroc_std,test_f1,test_auroc
0,main_model_comparison,XGBoost,time_weather_history,26,0.827265,0.016447,0.900611,0.013745,0.834769,0.906898
1,main_model_comparison,LogisticRegression,time_weather_history,26,0.771747,0.033031,0.891258,0.011619,0.807537,0.896619


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp1_main_model_comparison.csv
已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp1_main_model_comparison_cv_folds.csv


## 15. 實驗 2：特徵 Ablation

比較三組特徵：

- `Time-only`
- `Time + Weather`
- `Time + Weather + History`

模型固定比較：
- Logistic Regression
- XGBoost

這組實驗主要回答：**不同資訊來源對事故風險預測是否有幫助？**

In [23]:
ablation_models = {
    "LogisticRegression": lambda feats: build_lr_pipeline(feats, class_weight=None),
    "XGBoost": lambda feats: build_xgb_pipeline(feats, use_balancing=False),
}

ablation_results_df, ablation_cv_folds_df = run_experiment_suite(
    model_dict=ablation_models,
    feature_sets_dict=feature_sets,
    train_df=train_df,
    test_df=test_df,
    output_prefix="feature_ablation",
)

ablation_results_path = OUT_DIR / "exp2_feature_ablation.csv"
ablation_cv_path = OUT_DIR / "exp2_feature_ablation_cv_folds.csv"
ablation_results_df.to_csv(ablation_results_path, index=False, encoding="utf-8-sig")
ablation_cv_folds_df.to_csv(ablation_cv_path, index=False, encoding="utf-8-sig")

display(ablation_results_df)
print("已輸出：", ablation_results_path)
print("已輸出：", ablation_cv_path)

,experiment,model,feature_set,n_features,cv_f1_mean,cv_f1_std,cv_auroc_mean,cv_auroc_std,test_f1,test_auroc
0,feature_ablation,XGBoost,time_only,8,0.826539,0.019647,0.906260,0.011821,0.837419,0.912488
1,feature_ablation,XGBoost,time_weather_history,26,0.827265,0.016447,0.900611,0.013745,0.834769,0.906898
2,feature_ablation,XGBoost,time_weather,18,0.828616,0.015815,0.902482,0.011141,0.832253,0.908244
3,feature_ablation,LogisticRegression,time_weather_history,26,0.771747,0.033031,0.891258,0.011619,0.807537,0.896619
4,feature_ablation,LogisticRegression,time_only,8,0.763728,0.029800,0.867930,0.008953,0.794061,0.864867
5,feature_ablation,LogisticRegression,time_weather,18,0.753414,0.030890,0.865112,0.011007,0.790862,0.864974


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp2_feature_ablation.csv
已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp2_feature_ablation_cv_folds.csv


## 16. 實驗 3：類別不平衡處理

比較四種設定：

- Logistic Regression without balancing
- Logistic Regression + `class_weight='balanced'`
- XGBoost without balancing
- XGBoost + `scale_pos_weight`

> 註：XGBoost 並沒有 `class_weight='balanced_subsample'`。
> 因此這裡改用同樣是為了解決不平衡分類問題、且是 XGBoost 官方常用做法的 `scale_pos_weight = negative / positive`。

In [24]:
imbalance_feature_sets = {
    "time_weather_history": feature_sets["time_weather_history"]
}

imbalance_models = {
    "LR_no_balance": lambda feats: build_lr_pipeline(feats, class_weight=None),
    "LR_balanced": lambda feats: build_lr_pipeline(feats, class_weight="balanced"),
    "XGB_no_balance": lambda feats: build_xgb_pipeline(feats, use_balancing=False),
    "XGB_scale_pos_weight": lambda feats: build_xgb_pipeline(feats, use_balancing=True),
}

imbalance_results_df, imbalance_cv_folds_df = run_experiment_suite(
    model_dict=imbalance_models,
    feature_sets_dict=imbalance_feature_sets,
    train_df=train_df,
    test_df=test_df,
    output_prefix="imbalance_handling",
)

imbalance_results_path = OUT_DIR / "exp3_imbalance_handling.csv"
imbalance_cv_path = OUT_DIR / "exp3_imbalance_handling_cv_folds.csv"
imbalance_results_df.to_csv(imbalance_results_path, index=False, encoding="utf-8-sig")
imbalance_cv_folds_df.to_csv(imbalance_cv_path, index=False, encoding="utf-8-sig")

display(imbalance_results_df)
print("已輸出：", imbalance_results_path)
print("已輸出：", imbalance_cv_path)

print("train scale_pos_weight =", round(compute_scale_pos_weight(train_df['label']), 4))

,experiment,model,feature_set,n_features,cv_f1_mean,cv_f1_std,cv_auroc_mean,cv_auroc_std,test_f1,test_auroc
0,imbalance_handling,XGB_scale_pos_weight,time_weather_history,26,0.825769,0.014436,0.900827,0.012474,0.835191,0.907112
1,imbalance_handling,XGB_no_balance,time_weather_history,26,0.827265,0.016447,0.900611,0.013745,0.834769,0.906898
2,imbalance_handling,LR_balanced,time_weather_history,26,0.777776,0.026868,0.891114,0.011558,0.810522,0.896370
3,imbalance_handling,LR_no_balance,time_weather_history,26,0.771747,0.033031,0.891258,0.011619,0.807537,0.896619


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp3_imbalance_handling.csv
已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp3_imbalance_handling_cv_folds.csv
train scale_pos_weight = 1.0875


## 17. 實驗 4：資料量實驗

使用訓練集的前段時間資料做：

- 25%
- 50%
- 75%
- 100%

這樣可以觀察：
- 資料量增加時，F1 / AUROC 是否穩定提升
- Logistic Regression 與 XGBoost 對資料量的敏感度是否不同

這裡同樣固定使用最完整特徵組：
- `Time + Weather + History`

In [25]:
def get_prefix_train_subset(train_df, ratio):
    df = train_df.sort_values(["date", "district", "slot_id"]).reset_index(drop=True)
    n = max(1, int(len(df) * ratio))
    subset = df.iloc[:n].copy()
    return subset

learning_rows = []
learning_fold_rows = []
learning_ratios = [0.25, 0.50, 0.75, 1.00]
learning_models = {
    "LogisticRegression": lambda feats: build_lr_pipeline(feats, class_weight=None),
    "XGBoost": lambda feats: build_xgb_pipeline(feats, use_balancing=False),
}
learning_features = feature_sets["time_weather_history"]

for ratio in learning_ratios:
    sub_train = get_prefix_train_subset(train_df, ratio)
    ratio_label = f"{int(ratio * 100)}%"

    # 為了避免樣本太少時 split 失敗，動態決定 n_splits
    n_splits = 5
    if len(sub_train) < 100:
        n_splits = 3

    for model_name, model_builder in learning_models.items():
        cv_fold_df, cv_summary = time_series_cv_eval(
            model_builder=model_builder,
            feature_list=learning_features,
            train_df=sub_train,
            target_col="label",
            n_splits=n_splits,
        )
        holdout_metrics, _ = holdout_eval(
            model_builder=model_builder,
            feature_list=learning_features,
            train_df=sub_train,
            test_df=test_df,
            target_col="label",
        )

        row = {
            "train_ratio": ratio,
            "train_ratio_label": ratio_label,
            "train_rows": len(sub_train),
            "train_start": str(sub_train["date"].min().date()),
            "train_end": str(sub_train["date"].max().date()),
            "model": model_name,
            **cv_summary,
            "test_f1": holdout_metrics["f1"],
            "test_auroc": holdout_metrics["auroc"],
        }
        learning_rows.append(row)

        cv_fold_df = cv_fold_df.copy()
        cv_fold_df["train_ratio"] = ratio
        cv_fold_df["train_ratio_label"] = ratio_label
        cv_fold_df["model"] = model_name
        learning_fold_rows.append(cv_fold_df)

learning_curve_results_df = pd.DataFrame(learning_rows).sort_values(
    ["model", "train_ratio"]
).reset_index(drop=True)
learning_curve_folds_df = pd.concat(learning_fold_rows, ignore_index=True)

learning_curve_path = OUT_DIR / "exp4_learning_curve.csv"
learning_curve_cv_path = OUT_DIR / "exp4_learning_curve_cv_folds.csv"
learning_curve_results_df.to_csv(learning_curve_path, index=False, encoding="utf-8-sig")
learning_curve_folds_df.to_csv(learning_curve_cv_path, index=False, encoding="utf-8-sig")

display(learning_curve_results_df)
print("已輸出：", learning_curve_path)
print("已輸出：", learning_curve_cv_path)

,train_ratio,train_ratio_label,train_rows,train_start,train_end,model,cv_f1_mean,cv_f1_std,cv_auroc_mean,cv_auroc_std,test_f1,test_auroc
0,0.25,25%,2196,2024-01-01,2024-04-01,LogisticRegression,0.802111,0.046146,0.845717,0.066270,0.732600,0.886994
1,0.50,50%,4392,2024-01-01,2024-07-01,LogisticRegression,0.791933,0.033008,0.884853,0.015412,0.775722,0.891881
2,0.75,75%,6588,2024-01-01,2024-10-01,LogisticRegression,0.789881,0.025236,0.891336,0.006054,0.800844,0.895274
3,1.00,100%,8784,2024-01-01,2024-12-31,LogisticRegression,0.771747,0.033031,0.891258,0.011619,0.807537,0.896619
4,0.25,25%,2196,2024-01-01,2024-04-01,XGBoost,0.827412,0.033832,0.887876,0.021439,0.825390,0.897910
5,0.50,50%,4392,2024-01-01,2024-07-01,XGBoost,0.819452,0.013164,0.889633,0.007069,0.828717,0.903775
6,0.75,75%,6588,2024-01-01,2024-10-01,XGBoost,0.826423,0.005867,0.897338,0.004868,0.831490,0.903339
7,1.00,100%,8784,2024-01-01,2024-12-31,XGBoost,0.827265,0.016447,0.900611,0.013745,0.835064,0.907014


已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp4_learning_curve.csv
已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/exp4_learning_curve_cv_folds.csv


## 18. 基本資料觀察

這裡保留一些簡單觀察結果：

- 各行政區事故風險率
- 各時段事故風險率

In [26]:
district_rate = (
    final_df.groupby(["county_city", "district"], as_index=False)["label"]
            .mean()
            .rename(columns={"label": "accident_risk_rate"})
            .sort_values(["county_city", "accident_risk_rate"], ascending=[True, False])
)
district_rate.to_csv(OUT_DIR / "district_risk_rate.csv", index=False, encoding="utf-8-sig")

slot_rate = (
    final_df.groupby("slot_4h", as_index=False)["label"]
            .mean()
            .rename(columns={"label": "accident_risk_rate"})
)
slot_rate.to_csv(OUT_DIR / "slot_risk_rate.csv", index=False, encoding="utf-8-sig")

display(district_rate)
display(slot_rate)


,county_city,district,accident_risk_rate
1,新竹市,東區,0.589831
0,新竹市,北區,0.452348
2,新竹縣,竹北市,0.548792
3,新竹縣,竹東鎮,0.316689


,slot_4h,accident_risk_rate
0,00:00-03:59,0.000000
1,04:00-07:59,0.000000
2,08:00-11:59,0.649795
3,12:00-15:59,0.783516
4,16:00-19:59,0.840629
5,20:00-23:59,0.587551


## 19. Summary 輸出

最後把整體資料集資訊與四組實驗的輸出檔名整理成 json，方便後續寫報告。

In [27]:
summary_info = {
    "total_rows": len(final_df),
    "positive_rows": int(final_df["label"].sum()),
    "negative_rows": int((final_df["label"] == 0).sum()),
    "positive_rate": float(final_df["label"].mean()),
    "date_min": str(final_df["date"].min().date()),
    "date_max": str(final_df["date"].max().date()),
    "n_districts": int(final_df["district"].nunique()),
    "n_slots": int(final_df["slot_id"].nunique()),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "experiments": {
        "exp1_main_model_comparison": [
            "exp1_main_model_comparison.csv",
            "exp1_main_model_comparison_cv_folds.csv",
        ],
        "exp2_feature_ablation": [
            "exp2_feature_ablation.csv",
            "exp2_feature_ablation_cv_folds.csv",
        ],
        "exp3_imbalance_handling": [
            "exp3_imbalance_handling.csv",
            "exp3_imbalance_handling_cv_folds.csv",
        ],
        "exp4_learning_curve": [
            "exp4_learning_curve.csv",
            "exp4_learning_curve_cv_folds.csv",
        ],
    }
}

with open(OUT_DIR / "summary_info.json", "w", encoding="utf-8") as f:
    json.dump(summary_info, f, ensure_ascii=False, indent=2)

print("已輸出：", OUT_DIR / "summary_info.json")
print(summary_info)

已輸出： /home/dgx/Hong/school/AI/HW1/hsinchu_accident_risk/outputs/summary_info.json
{'total_rows': 17544, 'positive_rows': 8367, 'negative_rows': 9177, 'positive_rate': 0.47691518467852256, 'date_min': '2024-01-01', 'date_max': '2025-12-31', 'n_districts': 4, 'n_slots': 6, 'train_rows': 8784, 'test_rows': 8760, 'experiments': {'exp1_main_model_comparison': ['exp1_main_model_comparison.csv', 'exp1_main_model_comparison_cv_folds.csv'], 'exp2_feature_ablation': ['exp2_feature_ablation.csv', 'exp2_feature_ablation_cv_folds.csv'], 'exp3_imbalance_handling': ['exp3_imbalance_handling.csv', 'exp3_imbalance_handling_cv_folds.csv'], 'exp4_learning_curve': ['exp4_learning_curve.csv', 'exp4_learning_curve_cv_folds.csv']}}
